[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/13_Vector_Store.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 向量存储（Vector Store）—— 全面指南

## 概览

本 notebook 全面走读 Semantica 的 vector_store 模块，演示向量存储、相似度搜索、混合搜索和多后端支持，面向语义检索。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/vector_store/)

### 先看主线

12 篇把"意思"翻译成了坐标，但散装的向量用不起来。这一篇给向量建一座图书馆——能存、能搜、能滤、能管。九步分三段：

**存与搜（第 1–3 步）—— 基本功**

1. 第 1 步 存入：`VectorStore` 存向量带元数据
2. 第 2 步 搜相似：三种相似度度量各有适用
3. 第 3 步 建索引：数据量大了暴力搜扛不住——用近似换速度

**滤与排（第 4–6 步）—— 搜得准**

4. 第 4 步 混合搜索：语义相似 + 属性过滤，双条件夹逼
5. 第 5 步 元数据管理：元数据独立存放、按条件查、带 schema 校验
6. 第 6 步 结果融合：多路召回怎么合并排名（RRF）

**治与选（第 7–9 步）—— 管得住**

7. 第 7 步 命名空间：多租户隔离
8. 第 8 步 多后端：FAISS、Weaviate、Qdrant、Milvus 怎么选
9. 第 9 步 最佳实践：生产环境的经验数字

一句话抓住重点：**混合搜索是本篇的题眼**——纯向量搜索只认"意思近"，混合搜索再叠一层属性过滤，这是 RAG 检索精度的关键一招。

### 学习目标

完成本 notebook 后，你将能够：

- 存储和管理带元数据的向量
- 用不同度量做相似度搜索
- 使用向量与元数据结合的混合搜索
- 使用多种向量库后端（FAISS、Weaviate 等）
- 创建和管理向量索引
- 过滤和排序搜索结果
- 实现多租户的命名空间隔离

### 你将学到什么

| 组件 | 用途 | 适用场景 |
|-----------|---------|-------------|
| `VectorStore` | 主向量存储 | 所有向量操作 |
| `VectorIndexer` | 索引创建 | 性能优化 |
| `VectorRetriever` | 相似度搜索 | 找相似向量 |
| `HybridSearch` | 组合搜索 | 向量 + 元数据过滤 |
| `MetadataFilter` | 元数据过滤 | 按属性过滤 |
| `MetadataStore` | 元数据管理 | 存储向量元数据 |
| `NamespaceManager` | 多租户 | 隔离向量集合 |

---

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

---

In [ ]:
!pip install -q semantica


## 第 1 步：基础向量存储

图书馆开工，先学会上架：`VectorStore` 负责向量的存取。这里用 12 篇的 `TextEmbedder` 生成真实嵌入，100 条文本连同元数据一起入库。

### 什么是 VectorStore？

`VectorStore` 是向量操作的主接口：
- **存储**：带元数据存向量
- **搜索**：找相似向量
- **CRUD**：增删改查
- **多后端**：支持 FAISS、Weaviate、Qdrant、Milvus

In [ ]:
from semantica.vector_store import VectorStore
from semantica.embeddings import TextEmbedder
import numpy as np

# 1. Initialize Embedder (Select Provider & Model)
# You can choose 'sentence_transformers' or 'fastembed'
embedder = TextEmbedder(method="sentence_transformers", model_name="all-MiniLM-L6-v2")
dimension = embedder.get_embedding_dimension()

# 2. Create vector store
store = VectorStore(backend="faiss", dimension=dimension)

# 3. Generate Real Embeddings
texts = [f"Document {i}" for i in range(100)]
vectors = embedder.embed_batch(texts)

metadata = [
    {"text": txt, "category": "science" if i % 2 == 0 else "technology", "year": 2020 + (i % 4)}
    for i, txt in enumerate(texts)
]
# 4. Store vectors
vector_ids = store.store_vectors(vectors, metadata=metadata)

print(f"Stored {len(vector_ids)} vectors")
print(f"First 3 IDs: {vector_ids[:3]}")

## 第 2 步：相似度搜索

书上架了，学检索。怎么算"两个向量相近"？度量不止一种：

### 相似度度量

- **余弦相似度（Cosine）**：语义相似度的首选
- **L2 距离**：欧氏距离
- **点积（Dot Product）**：快，但要求向量先归一化

In [ ]:
from semantica.vector_store import VectorIndexer, FAISSStore 
import numpy as np 
# We use the first vector from the dataset as a sample query
if 'query_vector' not in locals():
    if 'vectors' in locals() and len(vectors) > 0:
        query_vector = vectors[0]
    else:
        # Fallback if vectors are also missing (safety check)
        query_vector = np.random.rand(dimension).astype('float32')

# Create indexer 
indexer = VectorIndexer(backend="faiss", dimension=dimension) 

# Create HNSW index for fast approximate search 
adapter = FAISSStore(dimension=dimension) 
index = adapter.create_index(index_type="hnsw", metric="L2", m=16) 

# Add vectors to index 
vectors_array = np.array(vectors).astype('float32') 
# FIX: Removed 'index' argument. The adapter uses its internal self.index 
adapter.add_vectors(vectors_array, ids=vector_ids) 

# Search using index 
query_array = np.array(query_vector).astype('float32') 
# FIX: Call search on the 'index' object directly, not the adapter 
distances, indices = index.search(query_array.reshape(1, -1), k=10) 

print(f"Index search found {len(indices[0])} results") 
print(f"Distances: {distances[0][:5]}")

## 第 3 步：向量索引

数据量一大，逐条比对的暴力搜就扛不住了。索引用"近似"换速度——牺牲一点点精度，换来数量级的提速。

### 索引类型（FAISS）

- **Flat**：精确搜索（暴力）
- **IVF**：倒排文件索引（近似）
- **HNSW**：分层图（速度与精度的最佳平衡）
- **PQ**：乘积量化（压缩存储）

In [ ]:
from semantica.vector_store import VectorIndexer, FAISSStore

# Get dimension from vectors to ensure consistency
dimension = len(vectors[0]) if len(vectors) > 0 else 384

# Create indexer
indexer = VectorIndexer(backend="faiss", dimension=dimension)

# Create HNSW index for fast approximate search
adapter = FAISSStore(dimension=dimension)
index = adapter.create_index(index_type="hnsw", metric="L2", m=16)

# Add vectors to index
vectors_array = np.array(vectors).astype('float32')
adapter.add_vectors(vectors_array, ids=vector_ids)

# Search using index
query_array = query_vector.astype('float32')
distances, indices = index.search(query_array.reshape(1, -1), k=10)

print(f"Index search found {len(indices[0])} results")
print(f"Distances: {distances[0][:5]}")

## 第 4 步：混合搜索

本篇题眼登场。纯向量搜索只认"意思近"——查"2021 年以后的科学文献"，向量不懂"2021 以后"是什么。混合搜索在语义相似之上再叠一层元数据过滤，双条件夹逼出更准的结果。

### 混合搜索的好处

- 向量搜索前先按元数据过滤
- 叠加多个搜索条件
- 结果更精确

In [ ]:
from semantica.vector_store import HybridSearch, MetadataFilter 
import numpy as np

# Create hybrid search 
hybrid_search = HybridSearch() 

# Create metadata filter 
filter = MetadataFilter() \
    .eq("category", "science") \
    .gt("year", 2021) 

# Perform hybrid search 
# Ensure vectors and metadata are available
if 'vectors' not in locals() or 'metadata' not in locals() or 'vector_ids' not in locals():
    print("Warning: vectors, metadata, or vector_ids are missing. Please run previous cells.")
else:
    hybrid_results = hybrid_search.search( 
        query_vector, 
        vectors, 
        metadata, 
        vector_ids, 
        filter=filter, 
        k=10 
    ) 
    
    print(f"Hybrid search found {len(hybrid_results)} results") 
    print("\nFiltered results (science, year > 2021):") 
    for i, result in enumerate(hybrid_results[:5], 1): 
        meta = result.get('metadata', {}) 
        print(f"{i}. Category: {meta.get('category')}, Year: {meta.get('year')}, Score: {result['score']:.3f}")

## 第 5 步：元数据管理

过滤要快，元数据就不能只挂在向量旁边——独立存放、独立查询，像图书馆的卡片目录。

### 元数据操作

- 为向量存元数据
- 按元数据条件查询
- 更新元数据
- schema 校验

In [ ]:
from semantica.vector_store import MetadataStore, MetadataSchema

# Create metadata store
meta_store = MetadataStore()

# Store metadata
for i, vec_id in enumerate(vector_ids[:10]):
    meta_store.store_metadata(vec_id, metadata[i])

# Query metadata
matching_ids = meta_store.query_metadata(
    {"category": "science"},
    operator="AND"
)

print(f"Found {len(matching_ids)} vectors with category='science'")

# Define schema for validation
schema = MetadataSchema({
    "text": {"type": str, "required": True},
    "category": {"type": str, "required": True},
    "year": {"type": int, "required": True}
})

# Validate metadata
is_valid = schema.validate(metadata[0])
print(f"\nMetadata validation: {is_valid}")

## 第 6 步：结果排序与融合

实际系统往往多路召回：向量一路、关键词一路、图谱一路。多份榜单怎么合成一份？排序融合解决它。

### 融合策略

- **倒数排名融合（RRF）**：合并多个排名列表
- **加权平均**：对不同来源的分数加权

In [ ]:
from semantica.vector_store import SearchRanker

# Create ranker with RRF strategy
ranker = SearchRanker(strategy="reciprocal_rank_fusion")

# Simulate multiple search results
results1 = [
    {"id": "vec_1", "score": 0.9},
    {"id": "vec_2", "score": 0.8},
    {"id": "vec_3", "score": 0.7}
]

results2 = [
    {"id": "vec_2", "score": 0.85},
    {"id": "vec_4", "score": 0.75},
    {"id": "vec_1", "score": 0.7}
]

# Fuse results using RRF
fused_results = ranker.rank([results1, results2], k=60)

print("Fused results using RRF:")
for i, result in enumerate(fused_results, 1):
    print(f"{i}. ID: {result['id']}, Fused Score: {result['score']:.3f}")

## 第 7 步：命名空间管理

进入"治与选"段。图书馆如果服务多家客户，各家藏书必须隔离——命名空间就是分柜管理。

### 命名空间特性

- 租户隔离
- 访问控制
- 按命名空间操作

In [ ]:
from semantica.vector_store import NamespaceManager

# Create namespace manager
ns_manager = NamespaceManager()

# Create namespaces for different tenants
ns1 = ns_manager.create_namespace("tenant1", "Tenant 1 vectors")
ns2 = ns_manager.create_namespace("tenant2", "Tenant 2 vectors")

# Add vectors to namespaces
for i in range(5):
    ns_manager.add_vector_to_namespace(f"t1_vec_{i}", "tenant1")
    ns_manager.add_vector_to_namespace(f"t2_vec_{i}", "tenant2")

# Get namespace vectors
tenant1_vectors = ns_manager.get_namespace_vectors("tenant1")
tenant2_vectors = ns_manager.get_namespace_vectors("tenant2")

print(f"Tenant 1: {len(tenant1_vectors)} vectors")
print(f"Tenant 2: {len(tenant2_vectors)} vectors")

# Set access control
ns1.set_access_control("user1", ["read", "write"])
ns1.set_access_control("user2", ["read"])

print(f"\nUser1 can write: {ns1.has_permission('user1', 'write')}")
print(f"User2 can write: {ns1.has_permission('user2', 'write')}")

## 第 8 步：多后端支持

一座图书馆可以有不同的建法。各后端定位不同，按需选型：

### 支持的后端

| 后端 | 类型 | 最适合 |
|---------|------|----------|
| FAISS | 本地 | 开发、小数据集 |
| Weaviate | 自托管 | schema 感知存储 |
| Qdrant | 自托管 | 高性能 |
| Milvus | 云/自托管 | 大规模 |

## 第 9 步：最佳实践

### 性能技巧

1. **向量归一化**：用余弦相似度就必须归一化
2. **用 HNSW**：速度与精度的最佳平衡
3. **批量操作**：按批处理（100-1000 条）
4. **先过滤**：元数据过滤放在向量搜索之前

### 后端选择

- **开发**：FAISS（本地、快）
- **生产**：Weaviate/Qdrant（可扩展、自托管）
- **自建机房**：Qdrant 或 Milvus（可控、高性能）
- **富元数据**：Weaviate（schema 感知）

### 索引配置

- **小数据集（<10K）**：Flat 索引
- **中等数据集（10K-1M）**：HNSW
- **大数据集（>1M）**：IVF + PQ

## 小结

### 回顾主线

图书馆三段走完：**存与搜**（VectorStore 上架、度量检索、索引提速）是基本功；**滤与排**（混合搜索、元数据目录、RRF 融合）把"搜到"变成"搜准"；**治与选**（命名空间隔离、多后端、经验数字）让它能上生产。至此检索侧三件套集齐：11 篇切块、12 篇造向量、本篇存与搜——14 篇将转向另一条线：本体（Ontology）。

### 你学到了什么

在本 notebook 中，你学会了：

- 用 VectorStore 存储和搜索向量
- 创建索引优化性能
- 使用带元数据过滤的混合搜索
- 独立于向量管理元数据
- 对搜索结果排序融合
- 实现命名空间隔离
- 用便捷函数快速操作
- 使用多种后端适配器
- 应用生产环境最佳实践

### 核心要点

1. **多后端**：按需选择
2. **混合搜索**：向量 + 元数据才有精度
3. **索引**：选对索引类型才有性能
4. **元数据**：独立管理才有灵活性
5. **命名空间**：多租户必备

### 下一步

**延伸阅读**：
- [Vector Store API 参考](https://semantica.readthedocs.io/reference/vector_store/)
- [高级向量存储 notebook](../advanced/Advanced_Vector_Store_and_Search.ipynb)
- [嵌入生成](12_Embedding_Generation.ipynb)

---

**有疑问或问题？** 欢迎访问我们的 [GitHub 仓库](https://github.com/semantica-agi/semantica)或[文档](https://semantica.readthedocs.io)。